# Football Passing Networks — Stable lineups

Nominal rosters, every link from one pass, and a separate exact-active audit. Spain–Germany, Spain–France and Spain–England at EURO 2024.

Data: StatsBomb Open Data / hudl. ![StatsBomb](../assets/statsbomb-logo.png)

Own-team substitutions/dismissals change the nominal lineup. Temporary departures are disclosed without splitting the main roster: all nodes need not be on the pitch at every instant. Exact active intervals and extra time remain separate. Main figures require at least five minutes and 20 completed open-play passes; all intervals remain in CSV/GraphML.

## 1. Reconstruct on-field membership

- Starting XI supplies initial membership.
- Substitution removes the outgoing player and adds the replacement.
- Same-stoppage batches are atomic when no football event intervenes.
- Player Off/On changes the exact-active audit, not the nominal interval.
- Red/second-yellow dismissals remove a player without replacement.
- Tactical Shift without membership changes updates role references only.

Half-open event indices [start_index, end_index) assign passes. Period-relative times include stoppage time and exclude breaks. A substitution batch uses its first event as the boundary; opponent substitutions do not split the lineup.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
import pandas as pd
from IPython.display import display, Image, Markdown
from src.analysis import load_dataset
from src.lineups import compute_fixed_lineups, validate_fixed_lineups, export_fixed_lineups
dataset=load_dataset(ROOT)
result=compute_fixed_lineups(dataset)
validation=validate_fixed_lineups(dataset,result)
display(validation)
summary=pd.DataFrame([{k:s[k] for k in ['segment_id','team','opponent','scope','start_label','end_label','players','network_passes','duration_seconds','temporary_absence_seconds','display_eligible','display_exclusion_reason']} for s in result['segments']])
display(summary)
display(result['temporary_absences'])

## 2. Draw every link from one pass

Use completed open-play passes with identified recipients, including periods 3–4 separately. Width = 0.30 pt × pass count on a common scale. Nodes represent sent+received involvement.

All roster members appear. Filled nodes have completed-pass involvement; hollow nodes use other interval events or schematic role references. The CSV/GraphML position_source records this distinction. Small visual offsets separate overlapping labels; grey ticks mark original locations. Coordinates in data exports stay unchanged. Sparse intervals remain in tables and GraphML; different lineups are never merged to meet display thresholds.

In [ ]:
figures, validation=export_fixed_lineups(dataset,result,ROOT)
print(f"{len(result['segments'])} intervals; {len(figures)} PNG + {len(figures)} SVG figures.")
overview=[item for item in figures if item['kind']=='overview']
individual=[item for item in figures if item['kind']=='individual']
def show(items):
    for item in items:
        display(Markdown(f"**{item['title']}**"))
        display(Image(filename=str(ROOT/'results/fixed_lineups/figures'/item['file'])))

## 3. Selected stable lineups — team overview pages

### Spain–Germany: regulation and separate extra time

In [ ]:
show([f for f in overview if f['match_id']==3942226])

### Spain–France

In [ ]:
show([f for f in overview if f['match_id']==3942752])

### Spain–England

In [ ]:
show([f for f in overview if f['match_id']==3943043])

## 4. Eligible intervals as separate figures

These intervals meet the declared five-minute/20-pass thresholds. Figures show raw counts, not rates per minute. Temporary absences are labelled. Omitted intervals remain in the first table and HTML report.

In [ ]:
show([f for f in individual if f['segment_id'].startswith('3942226_')])

In [ ]:
show([f for f in individual if f['segment_id'].startswith('3942752_')])

In [ ]:
show([f for f in individual if f['segment_id'].startswith('3943043_')])

## 5. Verification and exports

Every completed pass belongs to exactly one interval, edge endpoints belong to its roster, and summing all interval edge weights reproduces the original network. Repeat the checks for exact active membership. Regulation retains 2,160 passes and extra time 241; synthetic tests cover substitution batches, returns and dismissals.

In [ ]:
display(validate_fixed_lineups(dataset,result))
display(pd.read_csv(ROOT/'results/fixed_lineups/tables/segments.csv')
        .groupby(['team','opponent','scope']).agg(intervals=('segment_id','size'),passes=('network_passes','sum')))
print('results/fixed_lineups/report.html')